# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

In [29]:
import importlib.util
import importlib.metadata
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch>=2.6,<3', '--index-url', 'https://download.pytorch.org/whl/cpu'])
packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<4',
    'matplotlib': 'matplotlib>=3.8,<4', 'transformers': 'transformers[sentencepiece]==5.18.0',
    'datasets': 'datasets>=3,<6', 'evaluate': 'evaluate>=0.4,<1',
    'gradio': 'gradio>=5,<7', 'tensorboard': 'tensorboard>=2.16,<3',
    'IPython': 'ipython>=8', 'sentencepiece': 'sentencepiece>=0.2',
    'zstandard': 'zstandard>=0.22', 'accelerate': 'accelerate>=1',
    'sacremoses': 'sacremoses>=0.1', 'sklearn': 'scikit-learn>=1.4,<2'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec('transformers') is not None:
    if importlib.metadata.version('transformers') != '5.18.0':
        missing.append('transformers[sentencepiece]==5.18.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import copy
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
from torch.utils.tensorboard import SummaryWriter
from IPython.display import display, Markdown

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_bpe(documents, vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=['<unk>', '<eos>'],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tokenizer.train_from_iterator(documents, trainer=trainer)
    return tokenizer


def encode_documents(documents, tokenizer, batch_size=128):
    chunks = []
    eos_id = tokenizer.token_to_id('<eos>')
    for start in range(0, len(documents), batch_size):
        encoded = tokenizer.encode_batch(documents[start:start + batch_size])
        chunks.extend(np.asarray(item.ids + [eos_id], dtype=np.int32) for item in encoded)
    return torch.from_numpy(np.concatenate(chunks).astype(np.int64))


def tokenizer_statistics(tokenizer, documents):
    counts = []
    unknown = 0
    token_count = 0
    unknown_id = tokenizer.token_to_id('<unk>')
    for start in range(0, len(documents), 128):
        for encoded in tokenizer.encode_batch(documents[start:start + 128]):
            counts.append(len(encoded.ids))
            unknown += sum(token == unknown_id for token in encoded.ids)
            token_count += len(encoded.ids)
    return {'vocab_size': tokenizer.get_vocab_size(), 'documents': len(counts),
            'mean_tokens': float(np.mean(counts)), 'unk_fraction': unknown / max(1, token_count)}


def sample_batch(source, context_len, batch_size, generator, device, return_indices=False):
    positions = torch.randint(len(source) - context_len, (batch_size,), generator=generator)
    x = torch.stack([source[position:position + context_len] for position in positions])
    y = torch.stack([source[position + 1:position + context_len + 1] for position in positions])
    if return_indices:
        return x.to(device), y.to(device), positions
    return x.to(device), y.to(device)


class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, context_len, dropout):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        length = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:length, :length] == 0, float('-inf'))
        return self.dropout(F.softmax(scores, dim=-1)) @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(embed_dim, embed_dim // n_heads,
            context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([head(x) for head in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(embed_dim, 4 * embed_dim), nn.ReLU(),
                                  nn.Linear(4 * embed_dim, embed_dim), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.feed_forward(self.norm2(x))


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, context_len, dropout):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout)
                                      for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, ids, targets=None):
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, ids, max_new_tokens=60, temperature=0.8, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(ids[:, -self.context_len:])
            logits = logits[:, -1] / temperature
            if top_k is not None:
                cutoff = torch.topk(logits, min(top_k, logits.shape[-1])).values[:, [-1]]
                logits = logits.masked_fill(logits < cutoff, float('-inf'))
            token = torch.multinomial(F.softmax(logits, dim=-1), 1)
            ids = torch.cat([ids, token], dim=1)
        return ids


MODEL_KEYS = ['vocab_size', 'embed_dim', 'n_heads', 'n_layers', 'context_len', 'dropout']


def model_from_config(config):
    return MiniGPT(**{key: config[key] for key in MODEL_KEYS})


@torch.no_grad()
def estimate_loss(model, splits, config, device):
    was_training = model.training
    model.eval()
    rng = torch.Generator().manual_seed(SEED + 200)
    result = {}
    for name, source in splits.items():
        losses = []
        for _ in range(config['eval_iters']):
            x, y = sample_batch(source, config['context_len'], config['batch_size'], rng, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        result[name] = float(np.mean(losses))
    model.train(was_training)
    return result


def train_minigpt(name, config, splits, tokenizer, artifact_dir, device):
    cached_summary = artifact_dir / 'runs' / name / 'summary.json'
    if os.environ.get('LAB3_REUSE_ARTIFACTS') == '1' and cached_summary.exists():
        cached = json.loads(cached_summary.read_text())
        if all(cached.get(key) == value for key, value in config.items()):
            print('Завантажено збережений прогін:', name)
            return cached, pd.read_csv(cached_summary.parent / 'history.csv')
    set_seed()
    model = model_from_config(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
    training_rng = torch.Generator().manual_seed(SEED + 100)
    run_path = artifact_dir / 'runs' / name
    run_path.mkdir(parents=True, exist_ok=True)
    model_path = artifact_dir / 'models'
    model_path.mkdir(exist_ok=True)
    writer = SummaryWriter(str(run_path), purge_step=0)
    history = []
    best_val_loss = float('inf')
    best_iteration = 0
    evaluation_seconds = 0.0
    start_time = time.perf_counter()

    def evaluate(iteration):
        nonlocal best_val_loss, best_iteration, evaluation_seconds
        before = time.perf_counter()
        losses = estimate_loss(model, splits, config, device)
        evaluation_seconds += time.perf_counter() - before
        history.append({'iteration': iteration, 'train_loss': losses['train'],
                        'val_loss': losses['val']})
        writer.add_scalar('Loss/train', losses['train'], iteration)
        writer.add_scalar('Loss/val', losses['val'], iteration)
        checkpoint = {'config': dict(config), 'model_state_dict': {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
            'tokenizer_json': tokenizer.to_str(), 'iteration': iteration,
            'val_loss': losses['val'], 'seed': SEED}
        if iteration == 0:
            torch.save(checkpoint, model_path / f'{name}_first.pt')
        if losses['val'] < best_val_loss:
            best_val_loss, best_iteration = losses['val'], iteration
            torch.save(checkpoint, model_path / f'{name}.pt')
        if iteration % max(config['eval_interval'], 100) == 0 or iteration == config['max_iters']:
            print(f"{name}: iter={iteration}, train={losses['train']:.4f}, "
                  f"val={losses['val']:.4f}", flush=True)

    try:
        evaluate(0)
        for iteration in range(1, config['max_iters'] + 1):
            model.train()
            x, y = sample_batch(splits['train'], config['context_len'], config['batch_size'],
                             training_rng, device)
            _, loss = model(x, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if iteration % config['eval_interval'] == 0 or iteration == config['max_iters']:
                evaluate(iteration)
    finally:
        writer.close()
    total_seconds = time.perf_counter() - start_time
    history = pd.DataFrame(history)
    history.to_csv(run_path / 'history.csv', index=False)
    record = {'model': name, 'val_loss': best_val_loss, 'best_iteration': best_iteration,
              'last_iteration': config['max_iters'], 'parameters': sum(p.numel() for p in model.parameters()),
              'training_seconds': total_seconds, 'evaluation_seconds': evaluation_seconds,
              'eval_points': len(history), **config}
    (run_path / 'summary.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    del model, optimizer, loss, x, y
    clear_memory()
    return record, history


def generate_checkpoint(path, prompt, max_new_tokens=60, temperature=0.8, top_k=40):
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    tokenizer = Tokenizer.from_str(checkpoint['tokenizer_json'])
    model = model_from_config(checkpoint['config'])
    model.load_state_dict(checkpoint['model_state_dict'])
    ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long)
    set_seed(SEED + 7)
    generated = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    text = tokenizer.decode(generated[0].tolist())
    del model
    clear_memory()
    return text


def make_pipeline(task, model_id, **kwargs):
    from transformers import pipeline
    revisions = globals().get('MODEL_REVISIONS', {})
    return pipeline(task, model=model_id, revision=revisions.get(model_id, 'main'),
                    device=0 if torch.cuda.is_available() else -1, **kwargs)


def show_json(value):
    def convert(item):
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, torch.Tensor):
            return item.tolist()
        raise TypeError(type(item).__name__)
    print(json.dumps(value, ensure_ascii=False, indent=2, default=convert))

MODEL_REVISIONS = {'MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli': '0a71e92a985b6e1ad1828cf67ce9c459639c1dca', 'distilbert/distilgpt2': '2290a62682d06624634c1f46a6ad5be0f47f38aa', 'Qwen/Qwen3-0.6B': 'c1899de289a04d12100db370d81485cdf75e47ca', 'distilbert/distilroberta-base': 'fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b', 'dslim/bert-base-NER': 'd1a3e8f13f8c3566299d95fcfc9a8d2382a9affc', 'youscan/ukr-roberta-base': '8149bde480a6df9014aa934c3f8af30858dea5a6', 'google/tapas-base-finetuned-wtq': 'e3dde1905dea877b0df1a5c057533e48327dee77', 'tabularisai/multilingual-sentiment-analysis': 'eea032081f8d247b4303ef3565e7cec1b6f201c9', 'cardiffnlp/twitter-xlm-roberta-base-sentiment': 'f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8', 'ukr-models/uk-ner': '5dbb7e2ddde8efda84b8240504fe7ec049cbbb06'}

import io
import re
import tempfile
import unicodedata
import urllib.request
import zstandard
set_seed()
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_DIR = Path('/content/drive/MyDrive/ai_systems/lab3')
else:
    ARTIFACT_DIR = Path.cwd() if Path.cwd().name == 'lab3' else Path.cwd() / 'lab3'
for folder in ['models', 'runs', 'tokenizers', 'data', 'metrics']:
    (ARTIFACT_DIR / folder).mkdir(parents=True, exist_ok=True)
print('ПІБ: Некрасов Артур Дмитрович')
print('Пристрій:', device)
print('Артефакти:', ARTIFACT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ПІБ: Некрасов Артур Дмитрович
Пристрій: cpu
Артефакти: /content/drive/MyDrive/ai_systems/lab3


## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:

**1.** `train_from_iterator` очікує ітератор документів. Список віршів
передає кожен текст окремо. `train_from_iterator(corpus_text)` із рядком
передаватиме окремі символи: тренер не побачить послідовностей символів
усередині слів і не навчить потрібні злиття. Натомість
`train_from_iterator([corpus_text])` є коректним викликом для одного
документа. Якщо між віршами залишити пробіли/переноси, ByteLevel і далі
розділятиме слова; головна різниця — обсяг одного елемента в пам'яті
та втрата явних меж документів. Склеювання без роздільників також
створює штучні сусідства останнього й першого слова різних творів.

**2 — етапи.** Нормалізація впорядковує Unicode та потрібні правила
очищення тексту. Претокенізація розділяє його на попередні фрагменти,
наприклад слова й пунктуацію. Модель токенізації BPE/WordPiece/Unigram
перетворює ці фрагменти на токени та їхні ID. Постобробка додає службові
токени й формує потрібні входи моделі; під час пакетування також
утворюються attention mask та, за потреби, token type IDs.
Декодування виконує зворотне перетворення ID у текст.

**2 — спеціальні токени.** `<unk>` позначає фрагмент, який токенізатор
не може представити своїм словником. `<eos>` позначає кінець документа.
Це окремі ID, а не звичайні слова. Для byte-level BPE тут задано всі
256 базових байтів, тому довільний UTF-8 текст можна закодувати без `<unk>`.

**3.** На тому самому корпусі демо середня довжина вірша становить
**1027.80** токенів для словника 16000 та **1175.75**
для 6000. Частка `<unk>` — **0.00%** і **0.00%**.
Зменшення кількості злиттів залишає коротші підслова; базові байти
зберігаються, тому довжина зростає без втрати покриття алфавіту.
Однаковий CONTEXT_LEN охоплює менше символів і слів, а для проходу
того самого тексту потрібно більше токенів і навчальних вікон.

**4.** `<eos>` навчає модель межі документа. Без нього останні слова
одного вірша стають контекстом для початку наступного, а модель вчиться
на випадкових переходах між незалежними творами. За звичайною
конкатенацією вона не отримує окремого сигналу завершення документа.

**5.** `train_data` і `val_data` — одновимірні тензори `torch.int64`.
Кожен елемент — цілий ID токена зі словника BPE, включно з `<eos>`.
Це послідовності символічних індексів, а не embedding-вектори; embedding
модель обчислює сама. Послідовний зріз зберігає порядок токенів.

**6.** Перевірка виконується в 3.2 демо; її функцію та виконання на
випадковому батчі наведено нижче.

**7.** Рядок t нижньотрикутної маски дозволяє увагу до позицій j ≤ t.
Верхньотрикутна маска відкривала б майбутні токени. Без маски під час
teacher forcing модель бачила б наступні токени, зокрема правильну
відповідь, і могла б штучно знизити loss. Під час autoregressive-генерації
майбутніх токенів немає, тому така навчальна підказка зникає й якість
передбачення погіршується.

**8.** Найкращий і останній checkpoint збігаються, коли мінімальна
validation loss отримана на останньому оцінюванні, або їхні ваги однакові.
Якщо після мінімуму validation loss зростає, слід завантажувати раніший
checkpoint. У виконаному демо найкраща ітерація — **400**,
остання — **400**, val_loss = **7.6356**. **У цьому запуску найкращий checkpoint збігається з останнім.**

Для власних minigpt_v1 і minigpt_v2 найкращі checkpoint отримано на ітераціях 5000 і 5000 із 5000; вони також є останніми.

**10.** EVAL_INTERVAL змінено з 50 на 10: кількість оцінювань зросла
з **9** до **41**, додатково виконано **256** forward-батчів для оцінки
двох сплітів при EVAL_ITERS=4. Час оцінювання: **1.06 с**
і **3.91 с**; повний час: **22.74 с** та
**24.58 с**. Додатково витрачено приблизно 2.85 с саме на оцінювання; різниця повного часу — 1.84 с.
Частіша оцінка робить криву детальнішою,
але не гарантує меншого статистичного шуму: кількість батчів на одну
оцінку не змінилася. Тут для оцінки використовуються фіксовані вікна,
а генератор навчальних батчів окремий, тому зміна частоти не змінює
навчальну послідовність. Ще менший інтервал збільшує витрати при все
меншому прирості інформації про повільні зміни моделі.

**11.** Температура змінює логіти перед softmax:
$p_i(T)=\exp(z_i/T)/\sum_j\exp(z_j/T)$. За T < 1 розподіл стає
гострішим, за T > 1 — рівнішим. При T → ∞ усі скінченні логіти
після ділення прямують до 0, тож розподіл наближається до рівномірного
на дозволених токенах. За наявності top-p/top-k рівномірність стосується
лише множини кандидатів після відсікання. У початковому демо одночасно
змінюється й top-p, тому відмінності текстів не можна приписати тільки T.

**12.** Top-p (nucleus sampling) залишає найменший набір найімовірніших
токенів із сумарною ймовірністю щонайменше p, відкидає решту та
нормалізує розподіл повторно. На відміну від top-k, кількість залишених
кандидатів залежить від поточного розподілу ймовірностей.

**13.** Для Qwen3-0.6B запущено однаковий промпт із thinking=False/True
при однакових temperature=0.8, top_p=0.9, seed і ліміті 128 токенів.
Отримано **52 / 128** нових токенів,
час **11.50 / 28.22 с**.
**Режим True витратив 91 токенів на проміжне обчислення й дав коротку остаточну відповідь. Час зріс, але помітного поліпшення продовження історії немає: обидві відповіді переважно повторюють початок промпту.** Проміжний текст міркувань не виводиться; збережено
кількість токенів і остаточну відповідь.

In [30]:
def assert_batch_alignment(x, y, source, positions):
    assert x.shape == y.shape
    assert torch.equal(x[:, 1:], y[:, :-1])
    for row, position in enumerate(positions.tolist()):
        assert torch.equal(x[row].cpu(), source[position:position + x.shape[1]])
        assert torch.equal(y[row].cpu(), source[position + 1:position + x.shape[1] + 1])

test_source = torch.arange(200, dtype=torch.long)
test_x, test_y, test_positions = sample_batch(test_source, 16, 4,
    torch.Generator().manual_seed(SEED), device, True)
assert_batch_alignment(test_x, test_y, test_source, test_positions)
print('Assert-тест випадкового батчу: OK')


Assert-тест випадкового батчу: OK


---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

In [31]:
DATASET_ID = 'nyuuzyou/ukr-lit'
DATASET_REVISION = 'd59a871c8c30b98ab0eaa0545af31dc0861cdce0'
CORPUS_PATH = ARTIFACT_DIR / 'data/ukrainian_education.jsonl'
if not CORPUS_PATH.exists():
    url = f'https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/ukr-lit-0.jsonl.zst'
    selected = []
    seen = set()
    with urllib.request.urlopen(url, timeout=120) as response:
        reader = zstandard.ZstdDecompressor().stream_reader(response)
        for line in io.TextIOWrapper(reader, encoding='utf-8'):
            row = json.loads(line)
            text = re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', row.get('text') or '')).strip()
            letters = re.findall(r'[^\W\d_]', text)
            if not 350 <= len(text) <= 20000 or re.search('[ыэъёЫЭЪЁ]', text):
                continue
            if len(re.findall('[іїєґІЇЄҐ]', text)) < 8:
                continue
            if len(re.findall('[А-Яа-яІіЇїЄєҐґ]', text)) / max(1, len(letters)) < 0.85:
                continue
            text = re.sub(r'[^А-Яа-яІіЇїЄєҐґA-Za-z0-9\s.,;:!?\x27\"’«»()\-–—/%+=°]', ' ', text)
            text = re.sub(r'\s+', ' ', text).strip()
            if text.casefold() in seen:
                continue
            seen.add(text.casefold())
            selected.append({'text': text, 'title': row.get('title', ''), 'url': row.get('url', '')})
            if len(selected) == 1000:
                break
    with CORPUS_PATH.open('w', encoding='utf-8') as file:
        for row in selected:
            file.write(json.dumps(row, ensure_ascii=False) + '\n')
rows = [json.loads(line) for line in CORPUS_PATH.read_text(encoding='utf-8').splitlines()]
documents = [row['text'] for row in rows]
order = np.random.default_rng(SEED).permutation(len(documents))
split_index = int(len(documents) * 0.9)
train_docs = [documents[index] for index in order[:split_index]]
val_docs = [documents[index] for index in order[split_index:]]
corpus_text = '\n\n'.join(documents)
print('Корпус:', DATASET_ID)
print('Жанр: навчальні презентації українською мовою')
print('Документів:', len(documents), '| символів:', len(corpus_text))
print('Train / val:', len(train_docs), '/', len(val_docs))
display(pd.DataFrame(rows)[['title']].head())


Корпус: nyuuzyou/ukr-lit
Жанр: навчальні презентації українською мовою
Документів: 1000 | символів: 3657950
Train / val: 900 / 100


,title
0,Презентація на тему «Рене Декарт» (варіант 3)
1,Презентація на тему «Фенілкетонурія» (варіант 2)
2,Презентація на тему «Альтернативні види палива...
3,Презентація на тему «Графіка» (варіант 1)
4,Презентація на тему «Юрій Немирич»


Обрано навчальні презентації з колонки text. Залишено 1000 унікальних
текстів довжиною 350–20000 символів із переважанням української кирилиці;
прибрано порожні тексти, дублікати, сторонні символи та записи з літерами,
властивими російській орфографії. Це мовний фільтр, а не гарантія
досконалого визначення мови кожного запозичення. Жанр відрізняється
від поезії демо: більше термінів, визначень і чисел.

Документи детерміновано поділено 90/10 до навчання токенізаторів.
Усередині документів токени не перемішуються. Словники навчаються
тільки на train; validation не використовується для добору злиттів.

---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [32]:
from transformers import AutoTokenizer
parent_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B',
    revision=MODEL_REVISIONS['Qwen/Qwen3-0.6B'])
warmup_tokenizer = parent_tokenizer.train_new_from_iterator(
    (train_docs[start:start + 100] for start in range(0, len(train_docs), 100)), vocab_size=1000)
warmup_tokenizer.save_pretrained(ARTIFACT_DIR / 'tokenizers/qwen_retrained')
print('Новий словник на основі Qwen:', len(warmup_tokenizer))
print(warmup_tokenizer.tokenize('Український студент досліджує електромагнітні явища.'))
del parent_tokenizer, warmup_tokenizer
clear_memory()


Новий словник на основі Qwen: 1012
['Ð£', 'ÐºÑĢÐ°ÑĹÐ½', 'ÑģÑĮÐºÐ¸Ð¹', 'ĠÑģÑĤ', 'ÑĥÐ´', 'ÐµÐ½ÑĤ', 'ĠÐ´Ð¾', 'ÑģÐ»ÑĸÐ´', 'Ð¶Ñĥ', 'ÑĶ', 'ĠÐµÐ»Ðµ', 'ÐºÑĤ', 'ÑĢÐ¾Ð¼', 'Ð°', 'Ð³', 'Ð½Ñĸ', 'ÑĤÐ½Ñĸ', 'ĠÑı', 'Ð²Ð¸', 'Ñī', 'Ð°', '.']


In [33]:
small_tokenizers = {}
small_tokenizers['BPE'] = train_bpe(train_docs, 500)
wordpiece = Tokenizer(models.WordPiece(unk_token='<unk>', max_input_chars_per_word=200))
wordpiece.normalizer = normalizers.NFKC()
wordpiece.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
wordpiece.decoder = decoders.WordPiece(prefix='##')
wordpiece.train_from_iterator(train_docs, trainer=trainers.WordPieceTrainer(
    vocab_size=500, special_tokens=['<unk>', '<eos>'], show_progress=False))
small_tokenizers['WordPiece'] = wordpiece
unigram = Tokenizer(models.Unigram())
unigram.normalizer = normalizers.NFKC()
unigram.pre_tokenizer = pre_tokenizers.Metaspace(replacement='▁', prepend_scheme='always')
unigram.decoder = decoders.Metaspace(replacement='▁', prepend_scheme='always')
unigram.train_from_iterator(train_docs, trainer=trainers.UnigramTrainer(
    vocab_size=500, special_tokens=['<unk>', '<eos>'], unk_token='<unk>', show_progress=False))
small_tokenizers['Unigram'] = unigram
for name, tokenizer in small_tokenizers.items():
    tokenizer.save(str(ARTIFACT_DIR / 'tokenizers' / f'{name.lower()}_500.json'))
    print(name, 'vocab_size:', tokenizer.get_vocab_size())


BPE vocab_size: 500
WordPiece vocab_size: 500
Unigram vocab_size: 500


In [34]:
sample_text = ('Студент досліджує українську культуру. Навчання допомагає розуміти '
               'електромагнітні явища та міждисциплінарні зв’язки.')
selected_words = ['навчання', 'студент', 'електромагнітні', 'міждисциплінарність']
tokenizer_rows = []
word_rows = []
for name, tokenizer in small_tokenizers.items():
    encoded = tokenizer.encode(sample_text)
    tokenizer_rows.append({'algorithm': name, 'vocab_size': tokenizer.get_vocab_size(),
        'sample_tokens': len(encoded.ids), 'sample_unk': encoded.ids.count(tokenizer.token_to_id('<unk>'))})
    for word in selected_words:
        encoding = tokenizer.encode(word)
        word_rows.append({'algorithm': name, 'word': word,
                          'token_count': len(encoding.ids), 'tokens': ' | '.join(encoding.tokens)})
tokenizer_comparison = pd.DataFrame(tokenizer_rows)
word_comparison = pd.DataFrame(word_rows)
display(tokenizer_comparison)
display(word_comparison)
tokenizer_comparison.to_csv(ARTIFACT_DIR / 'data/tokenizer_comparison.csv', index=False)
word_comparison.to_csv(ARTIFACT_DIR / 'data/word_tokenization.csv', index=False)
shortest = tokenizer_comparison.loc[tokenizer_comparison['sample_tokens'].idxmin()]
display(Markdown(f"Найкоротша послідовність на обраному зразку: **{shortest['algorithm']}** "
    f"({int(shortest['sample_tokens'])} токенів). BPE нарощує словник частотними злиттями, "
    "WordPiece використовує підслова й жадібний пошук найдовшого збігу, Unigram "
    "обирає ймовірну сегментацію зі скорочуваного словника. Результат залежить "
    "від корпусу, алфавіту та претокенізації; byte-level BPE витрачає частину "
    "500 позицій на 256 базових байтів. Короткий результат із <unk> означав би "
    "втрату інформації, тому число невідомих токенів наведено окремо. "
    "Перенавчений токенізатор Qwen використано лише як вправу; готова Qwen "
    "далі завантажується зі своїм оригінальним токенізатором."))


,algorithm,vocab_size,sample_tokens,sample_unk
0,BPE,500,73,0
1,WordPiece,500,68,0
2,Unigram,500,66,0


,algorithm,word,token_count,tokens
0,BPE,навчання,5,Ð½Ð° | Ð² | Ñĩ | Ð°Ð½ | Ð½Ñı
1,BPE,студент,4,ÑģÑĤ | ÑĥÐ´ | ÐµÐ½ | ÑĤ
2,BPE,електромагнітні,10,Ðµ | Ð»Ðµ | ÐºÑĤ | ÑĢÐ¾Ð | ¼ | Ð° | Ð³ | Ð½Ñĸ ...
3,BPE,міждисциплінарність,12,Ð¼Ñĸ | Ð¶ | Ð´Ð¸ | Ñģ | ÑĨ | Ð¸ | Ð¿ | Ð»Ñĸ | ...
4,WordPiece,навчання,4,на | ##в | ##ча | ##ння
5,WordPiece,студент,5,ст | ##у | ##де | ##н | ##т
6,WordPiece,електромагнітні,9,е | ##ле | ##кт | ##ро | ##ма | ##г | ##ні | #...
7,WordPiece,міждисциплінарність,11,мі | ##ж | ##ди | ##с | ##ц | ##и | ##п | ##лі...
8,Unigram,навчання,4,▁на | в | ч | ання
9,Unigram,студент,5,▁ | сту | де | н | т


Найкоротша послідовність на обраному зразку: **Unigram** (66 токенів). BPE нарощує словник частотними злиттями, WordPiece використовує підслова й жадібний пошук найдовшого збігу, Unigram обирає ймовірну сегментацію зі скорочуваного словника. Результат залежить від корпусу, алфавіту та претокенізації; byte-level BPE витрачає частину 500 позицій на 256 базових байтів. Короткий результат із <unk> означав би втрату інформації, тому число невідомих токенів наведено окремо. Перенавчений токенізатор Qwen використано лише як вправу; готова Qwen далі завантажується зі своїм оригінальним токенізатором.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [35]:
gpt_tokenizer = train_bpe(train_docs, 2048)
gpt_tokenizer.save(str(ARTIFACT_DIR / 'tokenizers/minigpt_bpe_2048.json'))
train_data = encode_documents(train_docs, gpt_tokenizer)
val_data = encode_documents(val_docs, gpt_tokenizer)
splits = {'train': train_data, 'val': val_data}
print('Train:', train_data.shape, train_data.dtype)
print('Val:', val_data.shape, val_data.dtype)
print('EOS ID:', gpt_tokenizer.token_to_id('<eos>'))
print('Кількість EOS:', int((train_data == gpt_tokenizer.token_to_id('<eos>')).sum()),
      int((val_data == gpt_tokenizer.token_to_id('<eos>')).sum()))
print('Частка <unk> у val:', (val_data == gpt_tokenizer.token_to_id('<unk>')).float().mean().item())


Train: torch.Size([1266313]) torch.int64
Val: torch.Size([136218]) torch.int64
EOS ID: 1
Кількість EOS: 900 100
Частка <unk> у val: 0.0


In [ ]:
base_config = {'vocab_size': gpt_tokenizer.get_vocab_size(), 'embed_dim': 64,
    'n_heads': 4, 'n_layers': 1, 'context_len': 64, 'dropout': 0.1,
    'batch_size': 8, 'learning_rate': 0.001, 'max_iters': 5000,
    'eval_interval': 100, 'eval_iters': 6}
deep_config = dict(base_config)
deep_config['n_layers'] = 2
assert sum(base_config[key] != deep_config[key] for key in base_config) == 1
x_check, y_check, positions = sample_batch(train_data, 64, 8,
    torch.Generator().manual_seed(SEED), device, True)
assert_batch_alignment(x_check, y_check, train_data, positions)
minigpt_records = []
minigpt_histories = {}
for name, config in [('minigpt_v1', base_config), ('minigpt_v2', deep_config)]:
    record, history = train_minigpt(name, config, splits, gpt_tokenizer, ARTIFACT_DIR, device)
    minigpt_records.append(record)
    minigpt_histories[name] = history
minigpt_results = pd.DataFrame(minigpt_records)
minigpt_results.assign(section='own_minigpt').to_csv(ARTIFACT_DIR / 'data/minigpt_results.csv', index=False)
display(minigpt_results[['model', 'parameters', 'n_layers', 'val_loss', 'best_iteration', 'training_seconds']])


minigpt_v1: iter=0, train=7.7961, val=7.7701
minigpt_v1: iter=100, train=6.7793, val=6.7509
minigpt_v1: iter=200, train=6.7483, val=6.7158
minigpt_v1: iter=300, train=6.6966, val=6.6601
minigpt_v1: iter=400, train=6.6477, val=6.6164
minigpt_v1: iter=500, train=6.5754, val=6.5507
minigpt_v1: iter=600, train=6.4565, val=6.4544
minigpt_v1: iter=700, train=6.2840, val=6.3140
minigpt_v1: iter=800, train=6.1425, val=6.1819
minigpt_v1: iter=900, train=5.9925, val=6.0492
minigpt_v1: iter=1000, train=5.8903, val=5.9557
minigpt_v1: iter=1100, train=5.7769, val=5.8819
minigpt_v1: iter=1200, train=5.6962, val=5.7981
minigpt_v1: iter=1300, train=5.6159, val=5.7276
minigpt_v1: iter=1400, train=5.5422, val=5.6630
minigpt_v1: iter=1500, train=5.4934, val=5.6127
minigpt_v1: iter=1600, train=5.4335, val=5.5665
minigpt_v1: iter=1700, train=5.4073, val=5.5074
minigpt_v1: iter=1800, train=5.3628, val=5.4791
minigpt_v1: iter=1900, train=5.3436, val=5.4326
minigpt_v1: iter=2000, train=5.3071, val=5.4064
mini

minigpt_v1: iter=2300, train=5.2413, val=5.3377
minigpt_v1: iter=2400, train=5.2118, val=5.3094
minigpt_v1: iter=2500, train=5.1848, val=5.2954
minigpt_v1: iter=2600, train=5.1885, val=5.2738
minigpt_v1: iter=2700, train=5.1587, val=5.2511
minigpt_v1: iter=2800, train=5.1302, val=5.2356
minigpt_v1: iter=2900, train=5.1159, val=5.2264
minigpt_v1: iter=3000, train=5.1083, val=5.1996
minigpt_v1: iter=3100, train=5.0971, val=5.2065
minigpt_v1: iter=3200, train=5.0817, val=5.1830
minigpt_v1: iter=3300, train=5.0658, val=5.1622
minigpt_v1: iter=3400, train=5.0752, val=5.1620
minigpt_v1: iter=3500, train=5.0522, val=5.1518
minigpt_v1: iter=3600, train=5.0448, val=5.1398
minigpt_v1: iter=3700, train=5.0305, val=5.1325
minigpt_v1: iter=3800, train=5.0282, val=5.1213
minigpt_v1: iter=3900, train=5.0032, val=5.1080
minigpt_v1: iter=4000, train=5.0007, val=5.1040
minigpt_v1: iter=4100, train=4.9933, val=5.1068
minigpt_v1: iter=4200, train=4.9755, val=5.0837
minigpt_v1: iter=4300, train=4.9811, val

minigpt_v2: iter=1500, train=5.4178, val=5.5679
minigpt_v2: iter=1600, train=5.3656, val=5.5125


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, history) in zip(axes, minigpt_histories.items()):
    ax.plot(history['iteration'], history['train_loss'], label='train')
    ax.plot(history['iteration'], history['val_loss'], label='validation')
    ax.set_title(name)
    ax.set_xlabel('Ітерація')
    ax.set_ylabel('Cross-entropy')
    ax.legend()
plt.tight_layout()
plt.show()
OWN_PROMPT = 'Українська культура та освіта'
generation_rows = []
for record in minigpt_records:
    text = generate_checkpoint(ARTIFACT_DIR / 'models' / f"{record['model']}.pt", OWN_PROMPT,
                               max_new_tokens=70, temperature=0.8, top_k=40)
    generation_rows.append({'model': record['model'], 'prompt': OWN_PROMPT, 'text': text})
    print(record['model'], '| best iteration:', record['best_iteration'])
    print(text, '\n')
pd.DataFrame(generation_rows).to_csv(ARTIFACT_DIR / 'data/minigpt_generations.csv', index=False)
winner = min(minigpt_records, key=lambda record: record['val_loss'])
delta = minigpt_records[0]['val_loss'] - minigpt_records[1]['val_loss']
display(Markdown(f"Найкраща validation loss: **{winner['model']}**, "
    f"{winner['val_loss']:.4f}. Зміна n_layers з 1 до 2 дала різницю "
    f"{delta:+.4f} у бік зменшення loss. Словник, контекст, seed і навчальний "
    "бюджет однакові, тому в цьому порівнянні змінено лише глибину. "
    "Зразки обох моделей містять уривки правильних слів разом зі спотвореними словами й граматичними помилками; "
    "невелика перевага за loss не дала виразної переваги зв’язності на цьому промпті. "
    "Нижча token-level loss не є окремим вимірюванням змістовності тексту."))


---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [ ]:
del train_data, val_data, splits, corpus_text, rows, documents, train_docs, val_docs
del gpt_tokenizer, small_tokenizers, wordpiece, unigram, tokenizer
clear_memory()
print('Пам’ять звільнено для готових моделей.')


### 4.0. Запустіть приклади виконання різних NLP задач

In [ ]:
import transformers
from transformers import pipeline
print('Transformers:', transformers.__version__)
nlp_examples = []

def save_nlp_example(task, language, model_id, input_value, result):
    def convert(value):
        if isinstance(value, np.generic):
            return value.item()
        raise TypeError(type(value).__name__)
    item = json.loads(json.dumps({'task': task, 'language': language, 'model': model_id,
        'input': input_value, 'result': result}, ensure_ascii=False, default=convert))
    nlp_examples.append(item)
    show_json(result)


In [ ]:
model_id = 'tabularisai/multilingual-sentiment-analysis'
classifier = make_pipeline('sentiment-analysis', model_id)
english_sentences = ["I've been waiting for a HuggingFace course my whole life.", 'I hate this so much!']
save_nlp_example('sentiment-analysis', 'en', model_id, english_sentences, classifier(english_sentences))
del classifier
clear_memory()


In [ ]:
model_id = 'MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli'
classifier = make_pipeline('zero-shot-classification', model_id)
sentence = 'This is a course about the Transformers library'
result = classifier(sentence, candidate_labels=['education', 'politics', 'business'])
save_nlp_example('zero-shot-classification', 'en', model_id, sentence, result)
del classifier
clear_memory()


In [ ]:
model_id = 'distilbert/distilgpt2'
generator = make_pipeline('text-generation', model_id)
sentence = 'In this course, we will teach you how to'
set_seed()
result = generator(sentence, max_new_tokens=20, do_sample=True,
                   pad_token_id=generator.tokenizer.eos_token_id)
save_nlp_example('text-generation', 'en', model_id, sentence, result)
result = generator(sentence, max_length=30, num_return_sequences=2, do_sample=True,
                   pad_token_id=generator.tokenizer.eos_token_id)
show_json(result)
del generator
clear_memory()


In [ ]:
model_id = 'distilbert/distilroberta-base'
unmasker = make_pipeline('fill-mask', model_id)
sentence = 'This course will teach you all about <mask> models.'
save_nlp_example('fill-mask', 'en', model_id, sentence, unmasker(sentence, top_k=2))
del unmasker
clear_memory()


In [ ]:
model_id = 'dslim/bert-base-NER'
ner = make_pipeline('ner', model_id, aggregation_strategy='simple')
sentence = 'My name is Sylvain and I work at Hugging Face in Brooklyn.'
save_nlp_example('ner', 'en', model_id, sentence, ner(sentence))
del ner
clear_memory()


In [ ]:
model_id = 'google/tapas-base-finetuned-wtq'
question_answerer = make_pipeline('table-question-answering', model_id)
query = 'How many stars does the transformers repository have?'
table = {'Repository': ['Transformers', 'Datasets', 'Tokenizers'],
         'Stars': ['36542', '4512', '3934'], 'Contributors': ['651', '77', '34'],
         'Programming language': ['Python', 'Python', 'Rust, Python and NodeJS']}
result = question_answerer(query=query, table=table)
save_nlp_example('table-question-answering', 'en', model_id, {'query': query, 'table': table}, result)
del question_answerer
clear_memory()


### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [ ]:
model_id = 'tabularisai/multilingual-sentiment-analysis'
classifier = make_pipeline('sentiment-analysis', model_id)
sentences = ['Дякую за зрозумілу лекцію, тепер я впевнено розв’язую задачі.',
             'Заняття постійно скасовують, така організація мене обурює.']
save_nlp_example('sentiment-analysis', 'uk', model_id, sentences, classifier(sentences))
del classifier
clear_memory()


In [ ]:
model_id = 'MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli'
classifier = make_pipeline('zero-shot-classification', model_id)
sentence = 'Студенти створили програму, яка допомагає автоматично перевіряти навчальні роботи.'
result = classifier(sentence, candidate_labels=['освіта', 'спорт', 'економіка'],
                    hypothesis_template='Цей текст про {}.')
save_nlp_example('zero-shot-classification', 'uk', model_id, sentence, result)
del classifier
clear_memory()


In [48]:
model_id = 'Qwen/Qwen3-0.6B'
generator = make_pipeline('text-generation', model_id,
                          dtype=torch.float32 if device.type == 'cpu' else torch.float16)
sentence = 'Напиши українською два речення для оголошення про студентський клуб робототехніки.'
formatted = generator.tokenizer.apply_chat_template([{'role': 'user', 'content': sentence}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
result = generator(formatted, max_new_tokens=64, do_sample=False, return_full_text=False)
save_nlp_example('text-generation', 'uk', model_id, sentence, result)
del generator
clear_memory()


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[
  {
    "generated_text": "1. У студентському клубі робототехніки увідомляється про розмітку і зміст роботи, що допомагає узагальнити знання студентів.  \n"
  }
]


In [49]:
model_id = 'youscan/ukr-roberta-base'
unmasker = make_pipeline('fill-mask', model_id)
sentence = 'Студенти навчаються в <mask> та працюють над цікавими проєктами.'
save_nlp_example('fill-mask', 'uk', model_id, sentence, unmasker(sentence, top_k=5))
del unmasker
clear_memory()


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: youscan/ukr-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[
  {
    "score": 0.06182388588786125,
    "token": 10450,
    "token_str": " класі",
    "sequence": "Студенти навчаються в  класі та працюють над цікавими проєктами."
  },
  {
    "score": 0.050886135548353195,
    "token": 14434,
    "token_str": " інтернеті",
    "sequence": "Студенти навчаються в  інтернеті та працюють над цікавими проєктами."
  },
  {
    "score": 0.04648713022470474,
    "token": 4525,
    "token_str": " школі",
    "sequence": "Студенти навчаються в  школі та працюють над цікавими проєктами."
  },
  {
    "score": 0.03260503709316254,
    "token": 7869,
    "token_str": " університеті",
    "sequence": "Студенти навчаються в  університеті та працюють над цікавими проєктами."
  },
  {
    "score": 0.02468295954167843,
    "token": 33224,
    "token_str": " університетах",
    "sequence": "Студенти навчаються в  університетах та працюють над цікавими проєктами."
  }
]


In [50]:
model_id = 'ukr-models/uk-ner'
ner = make_pipeline('ner', model_id, aggregation_strategy='simple')
sentence = 'Олена Коваленко приїхала до Львова та відвідала Львівський національний університет.'
save_nlp_example('ner', 'uk', model_id, sentence, ner(sentence))
del ner
clear_memory()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[
  {
    "entity_group": "PER",
    "score": 0.9999728202819824,
    "word": "О",
    "start": 0,
    "end": 1
  },
  {
    "entity_group": "PER",
    "score": 0.999988853931427,
    "word": "лена Коваленко",
    "start": 1,
    "end": 15
  },
  {
    "entity_group": "LOC",
    "score": 0.9999794960021973,
    "word": "Львова",
    "start": 28,
    "end": 34
  },
  {
    "entity_group": "ORG",
    "score": 0.999981164932251,
    "word": "Львівськ",
    "start": 48,
    "end": 56
  },
  {
    "entity_group": "ORG",
    "score": 0.9806279540061951,
    "word": "ий національний університет.",
    "start": 56,
    "end": 84
  }
]


In [51]:
model_id = 'google/tapas-base-finetuned-wtq'
question_answerer = make_pipeline('table-question-answering', model_id)
query = 'Який університет має 1200 студентів?'
table = {'Університет': ['Київська політехніка', 'Львівська політехніка', 'Житомирська політехніка'],
         'Студентів': ['900', '1400', '1200'], 'Груп': ['30', '45', '40']}
result = question_answerer(query=query, table=table)
save_nlp_example('table-question-answering', 'uk', model_id, {'query': query, 'table': table}, result)
print('Контрольна відповідь за таблицею: Житомирська політехніка')
print('Мова fine-tuning TAPAS за model card: English')
(ARTIFACT_DIR / 'data/nlp_examples.json').write_text(
    json.dumps(nlp_examples, ensure_ascii=False, indent=2), encoding='utf-8')
del question_answerer
clear_memory()


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{
  "answer": "Житомирська політехніка",
  "coordinates": [
    [
      2,
      0
    ]
  ],
  "cells": [
    "Житомирська політехніка"
  ],
  "aggregator": "NONE"
}
Контрольна відповідь за таблицею: Житомирська політехніка
Мова fine-tuning TAPAS за model card: English


---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

In [52]:
import evaluate

for metric_name in ['accuracy', 'f1']:
    metric_dir = ARTIFACT_DIR / 'metrics' / metric_name
    metric_dir.mkdir(parents=True, exist_ok=True)
    metric_path = metric_dir / f'{metric_name}.py'
    if not metric_path.exists():
        urllib.request.urlretrieve(
            f'https://huggingface.co/spaces/evaluate-metric/{metric_name}/resolve/main/{metric_name}.py', metric_path)
accuracy_metric = evaluate.load(str(ARTIFACT_DIR / 'metrics/accuracy'))
f1_metric = evaluate.load(str(ARTIFACT_DIR / 'metrics/f1'))

test_examples = [
    ('Пояснення викладача були зрозумілими, я нарешті розібрався з темою.', 1),
    ('Дякую за цікаві практичні заняття та корисні поради.', 1),
    ('Мені сподобалося працювати в цій команді, усі підтримували одне одного.', 1),
    ('Навіть складні задачі стали посильними завдяки чудовим прикладам.', 1),
    ('Я задоволений курсом і радий, що записався на нього.', 1),
    ('Дуже приємно отримувати швидкий і доброзичливий зворотний зв’язок.', 1),
    ('Попри дрібні недоліки, навчання залишило гарне враження.', 1),
    ('Лабораторія добре обладнана, працювати тут одне задоволення.', 1),
    ('Пояснення спочатку здалося складним, але вдалі приклади все прояснили.', 1),
    ('Цей викладач надихає вчитися, із задоволенням прийду знову.', 1),
    ('Розклад змінюють без попередження, це дуже дратує.', 0),
    ('Завдання сформульоване незрозуміло, я розчарований організацією.', 0),
    ('Після кількох звернень мені так і не відповіли, це неприйнятно.', 0),
    ('В аудиторії холодно й шумно, навчатися тут некомфортно.', 0),
    ('Перевірка робіт була несправедливою, результат мене обурив.', 0),
    ('Я шкодую про витрачений час: корисних пояснень майже не було.', 0),
    ('Обіцяної підтримки не отримав, тому курсом незадоволений.', 0),
    ('Техніка постійно ламається, а проблему ніхто не вирішує.', 0),
    ('Не можу назвати таку організацію задовільною, усе доводиться з’ясовувати самому.', 0),
    ('Черги й постійні затримки зіпсували мені враження від навчання.', 0),
]
dev_examples = [
    ('Заняття виявилося захопливим, дякую за уважне ставлення.', 1),
    ('Нова система навчання зручна, користуватися нею приємно.', 1),
    ('Завдяки детальному поясненню я успішно завершив проєкт.', 1),
    ('Мені подобається доброзичлива атмосфера на курсі.', 1),
    ('Хоча було нелегко, результатом і підтримкою я задоволений.', 1),
    ('Практичне заняття знову скасували, це викликає обурення.', 0),
    ('Допомоги так і не дочекався, сервісом дуже розчарований.', 0),
    ('Незрозумілі вимоги й грубе ставлення зіпсували весь досвід.', 0),
    ('Не рекомендую цей курс: організація вкрай погана.', 0),
    ('Оцінку поставили без пояснень, така несправедливість дратує.', 0),
]
examples_frame = pd.DataFrame([
    {'split': split, 'text': text, 'label': label}
    for split, examples in [('validation', dev_examples), ('comparison', test_examples)]
    for text, label in examples])
examples_frame.to_csv(ARTIFACT_DIR / 'data/sentiment_examples.csv', index=False)
display(examples_frame)


def polar_scores(predictions):
    result = []
    for row in predictions:
        scores = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
        for entry in row:
            label = entry['label'].lower().replace('_', ' ')
            if 'positive' in label:
                scores['positive'] += float(entry['score'])
            elif 'negative' in label:
                scores['negative'] += float(entry['score'])
            elif 'neutral' in label:
                scores['neutral'] += float(entry['score'])
        total = scores['positive'] + scores['negative']
        assert total > 0
        scores['polar_positive'] = scores['positive'] / total
        result.append(scores)
    return result


def classification_metrics(references, probabilities, threshold=0.5):
    predictions = [int(probability >= threshold) for probability in probabilities]
    return {**accuracy_metric.compute(predictions=predictions, references=references),
            'f1': f1_metric.compute(predictions=predictions, references=references, average='macro')['f1']}


comparison_models = ['tabularisai/multilingual-sentiment-analysis',
                     'cardiffnlp/twitter-xlm-roberta-base-sentiment']
prediction_cache = {}
comparison_rows = []
prediction_rows = []
for model_id in comparison_models:
    classifier = make_pipeline('sentiment-analysis', model_id)
    classifier([dev_examples[0][0]], top_k=None, truncation=True, max_length=192)
    dev_scores = polar_scores(classifier([text for text, _ in dev_examples],
        top_k=None, batch_size=4, truncation=True, max_length=192))
    durations = []
    for repeat in range(3):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        before = time.perf_counter()
        raw_predictions = classifier([text for text, _ in test_examples],
            top_k=None, batch_size=4, truncation=True, max_length=192)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        durations.append(time.perf_counter() - before)
    test_scores = polar_scores(raw_predictions)
    dev_probabilities = [row['polar_positive'] for row in dev_scores]
    test_probabilities = [row['polar_positive'] for row in test_scores]
    prediction_cache[model_id] = {'validation': dev_probabilities, 'test': test_probabilities}
    test_metrics = classification_metrics([label for _, label in test_examples], test_probabilities)
    dev_metrics = classification_metrics([label for _, label in dev_examples], dev_probabilities)
    comparison_rows.append({'model': model_id, 'threshold': 0.5, **test_metrics,
        'validation_f1': dev_metrics['f1'], 'validation_accuracy': dev_metrics['accuracy'],
        'inference_seconds': float(np.mean(durations)),
        'parameters': sum(parameter.numel() for parameter in classifier.model.parameters())})
    for split, examples, scores in [('validation', dev_examples, dev_scores), ('comparison', test_examples, test_scores)]:
        for (text, label), score in zip(examples, scores):
            prediction_rows.append({'model': model_id, 'split': split, 'text': text,
                                    'reference': label, **score})
    del classifier
    clear_memory()
model_comparison = pd.DataFrame(comparison_rows)
display(model_comparison)
pd.DataFrame(prediction_rows).to_csv(ARTIFACT_DIR / 'data/sentiment_predictions.csv', index=False)
selected_model_row = model_comparison.sort_values(
    ['f1', 'accuracy', 'inference_seconds'], ascending=[False, False, True]).iloc[0]
selected_model_id = selected_model_row['model']
print('Краща модель на порівняльному наборі:', selected_model_id)


,split,text,label
0,validation,"Заняття виявилося захопливим, дякую за уважне ...",1
1,validation,"Нова система навчання зручна, користуватися не...",1
2,validation,Завдяки детальному поясненню я успішно заверши...,1
3,validation,Мені подобається доброзичлива атмосфера на курсі.,1
4,validation,"Хоча було нелегко, результатом і підтримкою я ...",1
5,validation,"Практичне заняття знову скасували, це викликає...",0
6,validation,"Допомоги так і не дочекався, сервісом дуже роз...",0
7,validation,Незрозумілі вимоги й грубе ставлення зіпсували...,0
8,validation,Не рекомендую цей курс: організація вкрай погана.,0
9,validation,"Оцінку поставили без пояснень, така несправедл...",0


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,model,threshold,accuracy,f1,validation_f1,validation_accuracy,inference_seconds,parameters
0,tabularisai/multilingual-sentiment-analysis,0.5,0.95,0.949875,1.0,1.0,1.188892,135328517
1,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.5,1.00,1.000000,1.0,1.0,1.855209,278045955


Краща модель на порівняльному наборі: cardiffnlp/twitter-xlm-roberta-base-sentiment


Обрано sentiment-analysis: 20 власних порівняльних речень (10 positive, 10 negative)
та окремі 10 validation-речень. Клас positive об'єднує Positive і Very Positive,
negative — Negative і Very Negative. Для бінарної задачі використано
P(positive)/(P(positive)+P(negative)); neutral не є правильним класом
жодного речення цього набору. Метрики — Accuracy і macro-F1 через evaluate.

Швидкість виміряно після прогріву, без завантаження ваг: середнє трьох
обробок однакового набору 20 речень на тому самому пристрої з batch_size=4.
Модель обрана за macro-F1 на 20 порівняльних реченнях, далі за Accuracy
і часом інференсу. Поріг обирається на окремих 10 validation-реченнях.
Порівняльний набір бере участь у виборі моделі, тому його результат
є описом цього експерименту, а не незалежною фінальною test-оцінкою.

In [ ]:
tuning_rows = []
for threshold in [0.5, 0.35, 0.65, 0.8]:
    test_metrics = classification_metrics([label for _, label in test_examples],
        prediction_cache[selected_model_id]['test'], threshold)
    dev_metrics = classification_metrics([label for _, label in dev_examples],
        prediction_cache[selected_model_id]['validation'], threshold)
    tuning_rows.append({'model': selected_model_id, 'threshold': threshold, **test_metrics,
        'validation_f1': dev_metrics['f1'], 'validation_accuracy': dev_metrics['accuracy'],
        'inference_seconds': selected_model_row['inference_seconds'],
        'parameters': int(selected_model_row['parameters'])})
tuning_results = pd.DataFrame(tuning_rows)
display(tuning_results)
best_setting = tuning_results.assign(distance=(tuning_results['threshold'] - 0.5).abs()).sort_values(
    ['validation_f1', 'validation_accuracy', 'distance'], ascending=[False, False, True]).iloc[0]
selected_threshold = float(best_setting['threshold'])
all_attempts = pd.concat([model_comparison, tuning_results[tuning_results['threshold'] != 0.5]], ignore_index=True)
all_attempts.to_csv(ARTIFACT_DIR / 'data/ready_model_comparison.csv', index=False)
display(all_attempts)
observed_best = all_attempts.sort_values(['f1', 'accuracy'], ascending=False).iloc[0]
model_effect = model_comparison['f1'].max() - model_comparison['f1'].min()
threshold_effect = tuning_results['f1'].max() - tuning_results['f1'].min()
display(Markdown(f"На порівняльних 20 реченнях найбільша macro-F1: "
    f"**{observed_best['model']}**, threshold={observed_best['threshold']:.2f}, "
    f"F1={observed_best['f1']:.4f}, Accuracy={observed_best['accuracy']:.4f}. "
    f"Різниця між двома моделями за базового порога — {model_effect:.4f} F1; "
    f"діапазон зміни F1 від трьох змін порога — {threshold_effect:.4f}. "
    f"Для застосунку обрано **{selected_model_id}**, "
    f"за validation підібрано поріг **{selected_threshold:.2f}**. Це невеликий вручну складений "
    "набір, тому результат не є універсальним рейтингом моделей."))
selection = {'model': selected_model_id, 'revision': MODEL_REVISIONS[selected_model_id],
             'threshold': selected_threshold}
(ARTIFACT_DIR / 'data/ready_model_selection.json').write_text(json.dumps(selection, indent=2))


---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [ ]:
import gradio as gr

feedback_classifier = make_pipeline('sentiment-analysis', selected_model_id)


def feedback_queue(text, threshold):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines or len(lines) > 30:
        raise gr.Error('Введіть від 1 до 30 відгуків, кожен із нового рядка.')
    predictions = feedback_classifier(lines, top_k=None, batch_size=4,
                                       truncation=True, max_length=192)
    scores = polar_scores(predictions)
    rows = []
    for line, score in zip(lines, scores):
        positive = score['polar_positive'] >= float(threshold)
        neutral = score['neutral'] > max(score['positive'], score['negative'])
        sentiment = 'Нейтральний' if neutral else ('Позитивний' if positive else 'Негативний')
        rows.append({'Відгук': line, 'Тональність': sentiment,
            'Оцінка негативу': round(score['negative'], 4),
            'Потребує уваги': sentiment == 'Негативний'})
    frame = pd.DataFrame(rows).sort_values(['Потребує уваги', 'Оцінка негативу'],
                                          ascending=[False, False]).reset_index(drop=True)
    counts = frame['Тональність'].value_counts().reindex(['Негативний', 'Нейтральний', 'Позитивний'], fill_value=0)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(counts.index, counts.values, color=['#c44e52', '#8c8c8c', '#55a868'])
    ax.set_ylabel('Кількість відгуків')
    ax.set_title('Тональність зворотного зв’язку')
    ax.set_ylim(0, max(1, counts.max()) + 1)
    plt.tight_layout()
    plt.close(fig)
    summary = f"Відгуків: {len(frame)}. Потребують уваги: {int(frame['Потребує уваги'].sum())}."
    return frame, fig, summary


feedback_example = '\n'.join([test_examples[0][0], test_examples[10][0], test_examples[14][0]])
demo = gr.Interface(feedback_queue,
    inputs=[gr.Textbox(lines=8, value=feedback_example, label='Відгуки: один із нового рядка'),
            gr.Slider(0.2, 0.8, value=selected_threshold, step=0.05, label='Поріг позитивної оцінки')],
    outputs=[gr.Dataframe(label='Черга перегляду', interactive=False),
             gr.Plot(label='Огляд'), gr.Textbox(label='Підсумок')],
    title='Черга уваги для відгуків про навчання',
    description='Додайте до 30 відгуків. Негативні оцінки з’являться першими в черзі перегляду.',
    api_name='sort_feedback', flagging_mode='never', analytics_enabled=False)
example_frame, example_plot, example_summary = feedback_queue(feedback_example, selected_threshold)
display(example_frame)
print(example_summary)
if os.environ.get('LAB3_RUN_GRADIO', '1') == '1':
    demo.launch(share=IN_COLAB, inline=True, prevent_thread_lock=True, show_error=True)


In [ ]:
result_frames = [pd.read_csv(ARTIFACT_DIR / 'data/minigpt_results.csv'),
                 all_attempts.assign(section='ready_sentiment')]
demo_results_path = ARTIFACT_DIR / 'data/demo_results.csv'
if demo_results_path.exists():
    result_frames.insert(0, pd.read_csv(demo_results_path))
results = pd.concat(result_frames, ignore_index=True, sort=False)
results.to_csv(ARTIFACT_DIR / 'results.csv', index=False)
display(results[['section', 'model', 'val_loss', 'accuracy', 'f1', 'threshold', 'parameters']])
print('Збережено:', ARTIFACT_DIR / 'results.csv')


In [56]:
from tensorboard import program
tensorboard_instance = program.TensorBoard()
tensorboard_instance.configure(argv=['tensorboard', '--logdir', str(ARTIFACT_DIR / 'runs'),
    '--host', '127.0.0.1', '--port', '0', '--load_fast', 'false'])
tensorboard_url = tensorboard_instance.launch()
print('TensorBoard:', tensorboard_url)
if IN_COLAB:
    from google.colab import output
    from urllib.parse import urlparse
    output.serve_kernel_port_as_iframe(urlparse(tensorboard_url).port, height=600)
else:
    from IPython.display import IFrame
    display(IFrame(tensorboard_url, width='100%', height=600))


TensorBoard: http://127.0.0.1:32903/


<IPython.core.display.Javascript object>

---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```